# Building Performance by Net Rentable Area

This notebook generates charts showing building performance metrics (occupancy rates and rental rates) segmented by building size.

**Data Sources:**
- Office: `quarterly_report_data_office` (Supabase)
- Industrial: `quarterly_report_data_industrial` (Supabase)

**Filters:**
- `aquila_competitive_set = True`
- `building_status = 'Existing'`

**Charts Generated:**
1. Office Occupancy Rate by Building Size
2. Office Weighted Average Rent by Building Size
3. Industrial Occupancy Rate by Building Size
4. Industrial Weighted Average Rent by Building Size

In [ ]:
# Imports
from dotenv import load_dotenv
from aquila_graphing_tools import initialize_supabase_connection, aquila_styled_line_chart, AQUILA_COLORS, AQUILA_FONT
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Load environment variables
load_dotenv('aquila_graph.env')

# Initialize Supabase connection
supabase = initialize_supabase_connection()
print("Connected to Supabase")

## 1. Fetch Office Data

In [ ]:
# Fetch office data
print("Fetching office data...")
response_office = supabase.table('quarterly_report_data_office') \
    .select('*') \
    .eq('aquila_competitive_set', True) \
    .eq('building_status', 'Existing') \
    .execute()

df_office = pd.DataFrame(response_office.data)
print(f"Loaded {len(df_office):,} office records")
print(f"Columns: {df_office.columns.tolist()}")
print(f"\nSample data:")
print(df_office.head())

## 2. Fetch Industrial Data

In [ ]:
# Fetch industrial data
print("Fetching industrial data...")
response_industrial = supabase.table('quarterly_report_data_industrial') \
    .select('*') \
    .eq('aquila_competitive_set', True) \
    .eq('building_status', 'Existing') \
    .execute()

df_industrial = pd.DataFrame(response_industrial.data)
print(f"Loaded {len(df_industrial):,} industrial records")
print(f"Columns: {df_industrial.columns.tolist()}")
print(f"\nSample data:")
print(df_industrial.head())

## 3. Data Cleaning and Preparation

In [ ]:
# Clean office data
print("\n=== OFFICE DATA CLEANING ===")

# Identify the date column
date_columns = [col for col in df_office.columns if 'date' in col.lower() or 'quarter' in col.lower()]
print(f"Potential date columns: {date_columns}")

# Parse dates (assuming 'quarter' or 'report_date' column)
if 'quarter' in df_office.columns:
    df_office['date'] = pd.to_datetime(df_office['quarter'], errors='coerce')
elif 'report_date' in df_office.columns:
    df_office['date'] = pd.to_datetime(df_office['report_date'], errors='coerce')
else:
    # Try to find any date column
    for col in date_columns:
        df_office['date'] = pd.to_datetime(df_office[col], errors='coerce')
        if df_office['date'].notna().any():
            print(f"Using '{col}' as date column")
            break

# Convert numeric columns
df_office['rentable_building_area'] = pd.to_numeric(df_office['rentable_building_area'], errors='coerce')
df_office['occupancy_pct_total'] = pd.to_numeric(df_office['occupancy_pct_total'], errors='coerce')
df_office['costar_rental_rate'] = pd.to_numeric(df_office['costar_rental_rate'], errors='coerce')

# Remove rows with missing critical data
df_office_clean = df_office[
    df_office['date'].notna() & 
    df_office['rentable_building_area'].notna() & 
    (df_office['rentable_building_area'] > 0)
].copy()

print(f"Office records after cleaning: {len(df_office_clean):,}")
print(f"Date range: {df_office_clean['date'].min()} to {df_office_clean['date'].max()}")
print(f"\nRentable area statistics (Office):")
print(df_office_clean['rentable_building_area'].describe())

In [ ]:
# Clean industrial data
print("\n=== INDUSTRIAL DATA CLEANING ===")

# Identify the date column
date_columns = [col for col in df_industrial.columns if 'date' in col.lower() or 'quarter' in col.lower()]
print(f"Potential date columns: {date_columns}")

# Parse dates
if 'quarter' in df_industrial.columns:
    df_industrial['date'] = pd.to_datetime(df_industrial['quarter'], errors='coerce')
elif 'report_date' in df_industrial.columns:
    df_industrial['date'] = pd.to_datetime(df_industrial['report_date'], errors='coerce')
else:
    for col in date_columns:
        df_industrial['date'] = pd.to_datetime(df_industrial[col], errors='coerce')
        if df_industrial['date'].notna().any():
            print(f"Using '{col}' as date column")
            break

# Convert numeric columns
df_industrial['rentable_building_area'] = pd.to_numeric(df_industrial['rentable_building_area'], errors='coerce')
df_industrial['occupancy_pct_total'] = pd.to_numeric(df_industrial['occupancy_pct_total'], errors='coerce')
df_industrial['survey_rental_rate'] = pd.to_numeric(df_industrial['survey_rental_rate'], errors='coerce')

# Remove rows with missing critical data
df_industrial_clean = df_industrial[
    df_industrial['date'].notna() & 
    df_industrial['rentable_building_area'].notna() & 
    (df_industrial['rentable_building_area'] > 0)
].copy()

print(f"Industrial records after cleaning: {len(df_industrial_clean):,}")
print(f"Date range: {df_industrial_clean['date'].min()} to {df_industrial_clean['date'].max()}")
print(f"\nRentable area statistics (Industrial):")
print(df_industrial_clean['rentable_building_area'].describe())

## 4. Create Size Bins

In [ ]:
# Create 5 bins for office with rounded ranges
office_min = df_office_clean['rentable_building_area'].min()
office_max = df_office_clean['rentable_building_area'].max()
office_quartiles = df_office_clean['rentable_building_area'].quantile([0.2, 0.4, 0.6, 0.8]).values

print("\n=== OFFICE SIZE BINS ===")
print(f"Min: {office_min:,.0f} SF")
print(f"20th percentile: {office_quartiles[0]:,.0f} SF")
print(f"40th percentile: {office_quartiles[1]:,.0f} SF")
print(f"60th percentile: {office_quartiles[2]:,.0f} SF")
print(f"80th percentile: {office_quartiles[3]:,.0f} SF")
print(f"Max: {office_max:,.0f} SF")

# Round to create readable bins
def round_to_readable(value):
    """Round to nearest 5k, 10k, 25k, 50k, or 100k depending on magnitude"""
    if value < 10000:
        return round(value / 5000) * 5000
    elif value < 50000:
        return round(value / 10000) * 10000
    elif value < 100000:
        return round(value / 25000) * 25000
    else:
        return round(value / 50000) * 50000

office_bins = [
    0,
    round_to_readable(office_quartiles[0]),
    round_to_readable(office_quartiles[1]),
    round_to_readable(office_quartiles[2]),
    round_to_readable(office_quartiles[3]),
    float('inf')
]

office_labels = [
    f"0-{office_bins[1]/1000:.0f}k SF",
    f"{office_bins[1]/1000:.0f}k-{office_bins[2]/1000:.0f}k SF",
    f"{office_bins[2]/1000:.0f}k-{office_bins[3]/1000:.0f}k SF",
    f"{office_bins[3]/1000:.0f}k-{office_bins[4]/1000:.0f}k SF",
    f"{office_bins[4]/1000:.0f}k+ SF"
]

print(f"\nOffice bins: {office_bins}")
print(f"Office labels: {office_labels}")

df_office_clean['size_bin'] = pd.cut(
    df_office_clean['rentable_building_area'],
    bins=office_bins,
    labels=office_labels,
    include_lowest=True
)

In [ ]:
# Create 5 bins for industrial with rounded ranges
industrial_min = df_industrial_clean['rentable_building_area'].min()
industrial_max = df_industrial_clean['rentable_building_area'].max()
industrial_quartiles = df_industrial_clean['rentable_building_area'].quantile([0.2, 0.4, 0.6, 0.8]).values

print("\n=== INDUSTRIAL SIZE BINS ===")
print(f"Min: {industrial_min:,.0f} SF")
print(f"20th percentile: {industrial_quartiles[0]:,.0f} SF")
print(f"40th percentile: {industrial_quartiles[1]:,.0f} SF")
print(f"60th percentile: {industrial_quartiles[2]:,.0f} SF")
print(f"80th percentile: {industrial_quartiles[3]:,.0f} SF")
print(f"Max: {industrial_max:,.0f} SF")

industrial_bins = [
    0,
    round_to_readable(industrial_quartiles[0]),
    round_to_readable(industrial_quartiles[1]),
    round_to_readable(industrial_quartiles[2]),
    round_to_readable(industrial_quartiles[3]),
    float('inf')
]

industrial_labels = [
    f"0-{industrial_bins[1]/1000:.0f}k SF",
    f"{industrial_bins[1]/1000:.0f}k-{industrial_bins[2]/1000:.0f}k SF",
    f"{industrial_bins[2]/1000:.0f}k-{industrial_bins[3]/1000:.0f}k SF",
    f"{industrial_bins[3]/1000:.0f}k-{industrial_bins[4]/1000:.0f}k SF",
    f"{industrial_bins[4]/1000:.0f}k+ SF"
]

print(f"\nIndustrial bins: {industrial_bins}")
print(f"Industrial labels: {industrial_labels}")

df_industrial_clean['size_bin'] = pd.cut(
    df_industrial_clean['rentable_building_area'],
    bins=industrial_bins,
    labels=industrial_labels,
    include_lowest=True
)

## 5. Calculate Weighted Metrics by Date and Size Bin

In [ ]:
# Office - Weighted occupancy rate
print("\n=== CALCULATING OFFICE WEIGHTED OCCUPANCY ===")

office_occ_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['occupancy_pct_total'].dropna(),
        weights=x.loc[x['occupancy_pct_total'].notna(), 'rentable_building_area']
    ) if len(x['occupancy_pct_total'].dropna()) > 0 else np.nan
).reset_index(name='weighted_occupancy_pct')

print(f"Office occupancy data shape: {office_occ_by_size.shape}")
print(office_occ_by_size.head())

In [ ]:
# Office - Weighted average rent
print("\n=== CALCULATING OFFICE WEIGHTED RENT ===")

office_rent_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['costar_rental_rate'].dropna(),
        weights=x.loc[x['costar_rental_rate'].notna(), 'rentable_building_area']
    ) if len(x['costar_rental_rate'].dropna()) > 0 else np.nan
).reset_index(name='weighted_avg_rent')

print(f"Office rent data shape: {office_rent_by_size.shape}")
print(office_rent_by_size.head())

In [ ]:
# Industrial - Weighted occupancy rate
print("\n=== CALCULATING INDUSTRIAL WEIGHTED OCCUPANCY ===")

industrial_occ_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['occupancy_pct_total'].dropna(),
        weights=x.loc[x['occupancy_pct_total'].notna(), 'rentable_building_area']
    ) if len(x['occupancy_pct_total'].dropna()) > 0 else np.nan
).reset_index(name='weighted_occupancy_pct')

print(f"Industrial occupancy data shape: {industrial_occ_by_size.shape}")
print(industrial_occ_by_size.head())

In [ ]:
# Industrial - Weighted average rent
print("\n=== CALCULATING INDUSTRIAL WEIGHTED RENT ===")

industrial_rent_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['survey_rental_rate'].dropna(),
        weights=x.loc[x['survey_rental_rate'].notna(), 'rentable_building_area']
    ) if len(x['survey_rental_rate'].dropna()) > 0 else np.nan
).reset_index(name='weighted_avg_rent')

print(f"Industrial rent data shape: {industrial_rent_by_size.shape}")
print(industrial_rent_by_size.head())

## 6. Generate Charts

In [ ]:
# Chart 1: Office Occupancy Rate by Building Size
fig_office_occ = aquila_styled_line_chart(
    office_occ_by_size,
    x='date',
    y='weighted_occupancy_pct',
    color='size_bin',
    title='Office Occupancy Rate by Building Size (Weighted by Rentable Area)'
)
fig_office_occ.update_yaxes(tickformat='.1%', title='Occupancy Rate')
fig_office_occ.update_xaxes(title='Quarter')
fig_office_occ.write_html('charts/office_occupancy_by_size.html')
print("Saved: charts/office_occupancy_by_size.html")
fig_office_occ.show()

In [ ]:
# Chart 2: Office Weighted Average Rent by Building Size
fig_office_rent = aquila_styled_line_chart(
    office_rent_by_size,
    x='date',
    y='weighted_avg_rent',
    color='size_bin',
    title='Office Weighted Average Rent by Building Size'
)
fig_office_rent.update_yaxes(tickprefix='$', tickformat=',.2f', title='Rent ($/SF)')
fig_office_rent.update_xaxes(title='Quarter')
fig_office_rent.write_html('charts/office_rent_by_size.html')
print("Saved: charts/office_rent_by_size.html")
fig_office_rent.show()

In [ ]:
# Chart 3: Industrial Occupancy Rate by Building Size
fig_industrial_occ = aquila_styled_line_chart(
    industrial_occ_by_size,
    x='date',
    y='weighted_occupancy_pct',
    color='size_bin',
    title='Industrial Occupancy Rate by Building Size (Weighted by Rentable Area)'
)
fig_industrial_occ.update_yaxes(tickformat='.1%', title='Occupancy Rate')
fig_industrial_occ.update_xaxes(title='Quarter')
fig_industrial_occ.write_html('charts/industrial_occupancy_by_size.html')
print("Saved: charts/industrial_occupancy_by_size.html")
fig_industrial_occ.show()

In [ ]:
# Chart 4: Industrial Weighted Average Rent by Building Size
fig_industrial_rent = aquila_styled_line_chart(
    industrial_rent_by_size,
    x='date',
    y='weighted_avg_rent',
    color='size_bin',
    title='Industrial Weighted Average Rent by Building Size'
)
fig_industrial_rent.update_yaxes(tickprefix='$', tickformat=',.2f', title='Rent ($/SF)')
fig_industrial_rent.update_xaxes(title='Quarter')
fig_industrial_rent.write_html('charts/industrial_rent_by_size.html')
print("Saved: charts/industrial_rent_by_size.html")
fig_industrial_rent.show()

## 7. Summary Statistics

In [ ]:
print("\n=== SUMMARY ===")
print(f"\nOffice:")
print(f"  - Total buildings tracked: {df_office_clean['building_id'].nunique() if 'building_id' in df_office_clean.columns else 'N/A'}")
print(f"  - Date range: {office_occ_by_size['date'].min().strftime('%Y-%m-%d')} to {office_occ_by_size['date'].max().strftime('%Y-%m-%d')}")
print(f"  - Size bins: {office_labels}")

print(f"\nIndustrial:")
print(f"  - Total buildings tracked: {df_industrial_clean['building_id'].nunique() if 'building_id' in df_industrial_clean.columns else 'N/A'}")
print(f"  - Date range: {industrial_occ_by_size['date'].min().strftime('%Y-%m-%d')} to {industrial_occ_by_size['date'].max().strftime('%Y-%m-%d')}")
print(f"  - Size bins: {industrial_labels}")

print("\nAll charts saved to charts/ directory!")